<a href="https://colab.research.google.com/github/ryanconquers/FlyrankAi_MLintern/blob/main/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
## Research Question

Which content pages should a reviewer look at first, given limited review capacity,
using only observable search/engagement signals (never the product's own decision flags)?

## Unit of Analysis

One row = one content item (`content_hash_id`), evaluated over a defined 90-day
observation window. Not a client, not a day — a page is the thing that gets
reviewed and (possibly) fixed.

## Output

A ranked queue: top-K content items, each with a numeric opportunity/risk score
and a reason code explaining *why* it surfaced (e.g. "high impressions, weak CTR
for its position tier" or "stale and still visible").

## The Action

A reviewer/content team spends their next sprint on the top N pages from the
queue instead of picking pages by gut feel or recency. The realistic action per
page: refresh, expand, protect, prune, or monitor.

## Cost of a Wrong Recommendation

- False positive (flagged but not actually worth fixing): wasted reviewer hours —
  cheap-ish, but compounds if it happens often, since capacity is the scarce resource.
- False negative (a genuinely declining, high-demand page never surfaces): real
  lost visibility/traffic compounding silently, since nobody looks at it until it's
  much worse. This is the more expensive error — asymmetric cost, which argues for
  optimizing recall on high-volume pages even at some precision cost.

## Why Data/ML Helps At All

A human reviewer can hold maybe 2-3 signals in their head at once (e.g. "old and
low position = bad"). The starter pipeline already shows a hand-written single-rule
baseline (Precision@50 = 0.24) being beaten by a random forest (Precision@50 = 0.74)
that combines many weak signals nonlinearly — evidence that the real pattern isn't
capturable by one or two hand-picked thresholds.

In [3]:
import pandas as pd
import os, subprocess
if not os.path.exists("flyrank-ml-internship-starter"):
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/flyrank-bih/flyrank-ml-internship-starter"], check=True)
os.chdir("flyrank-ml-internship-starter")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. How big is the eligible population? (matches starter's own filter)
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print(f"Eligible pages: {len(eligible):,} of {len(df):,} total "
      f"({len(eligible)/len(df):.1%})")

# 2. How much of the "problem" actually exists in this slice?
declining_pct = (eligible["trend_direction"] == "down").mean()
print(f"Share currently declining: {declining_pct:.1%}")

# 3. How big is the "stale but still visible" opportunity bucket?
stale_visible = eligible[
    (eligible["days_since_last_update"] >= 180) &
    (eligible["impressions_90d"] >= 500)
]
print(f"Stale-but-visible pages: {len(stale_visible):,} "
      f"({len(stale_visible)/len(eligible):.1%} of eligible)")

Eligible pages: 30,000 of 30,000 total (100.0%)
Share currently declining: 54.2%
Stale-but-visible pages: 17 (0.1% of eligible)
